# Physical models and the majorization criteria

This notebook explores the majorization / generalized-Klyshko criteria and
the classic analytical criteria, evaluated on Jaynes-Cummings statistics.
The QuTiP-based state generators are demonstrated at the end and are
**optional** (require `pip install .[physics]`).

In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.analytical.jaynes_cummings import Pjaycum
from src.analytical.criteria_majorization import (
    all_pairs_for_list_of_P, load_precomputed_pairs)
from src.analytical.criteria_classic import (
    klyshko, taylored, log01rule, analcrit2D, generalized_klyshko_first_Np)

## Majorization pairs

A classical distribution satisfies a polynomial inequality for every
majorization pair $(T^{(0)}, T^{(1)})$ with $T^{(0)}\succcurlyeq T^{(1)}$.
These can be generated on the fly or loaded from the precomputed archive.

In [ ]:
generated = all_pairs_for_list_of_P(indx=np.array([0, 1, 2, 3]))
precomputed = load_precomputed_pairs(np.array([0, 1, 2, 3]))
print('generated pairs   :', len(generated))
print('precomputed pairs :', None if precomputed is None else len(precomputed))

## Classic criteria on a Jaynes-Cummings grid

We build a $(\nu, \tau)$ grid of Jaynes-Cummings statistics and map the
fraction of nonclassical points found by each criterion.

In [ ]:
tau = np.linspace(0.01, 10.0, 200)
nu = np.linspace(0.01, 6.0, 200)
xtau, ynu = np.meshgrid(tau, nu)

NP = 6
Pall = np.stack([Pjaycum(n=n, nu=ynu, tau=xtau, pe=1.0/3.0) for n in range(NP)], axis=-1)

for name, fn in [('klyshko', klyshko), ('taylored', taylored),
                 ('log01rule', log01rule), ('analcrit2D', analcrit2D)]:
    _, frac = fn(Pall)
    print(f'{name:12s} nonclassical fraction = {frac:.3f}')

In [ ]:
nonclass, frac = analcrit2D(Pall)
plt.figure(figsize=(7, 4))
plt.pcolormesh(xtau, ynu, nonclass.astype(int), shading='auto', cmap='Greys')
plt.xlabel(r'$\tau$'); plt.ylabel(r'$\nu$')
plt.title(f'Analytical nonclassicality map (fraction={frac:.3f})')
plt.tight_layout(); plt.show()

## Optional: QuTiP state generators

The cell below builds a photon-added coherent state and reads off its
photon-number probabilities. It is skipped gracefully if QuTiP is not
installed.

In [ ]:
try:
    from src.analytical.models import pba_Pk_co
    pk = [float(pba_Pk_co(Ns=60, mu=1.0, p=0.3, k=k)) for k in range(5)]
    print('photon-added coherent P_k =', np.round(pk, 4))
except ImportError:
    print('QuTiP not installed; skipping state-generator demo.')